# <span style="color: black;"> MODIS_&_Wind_Resampled_Combined_Dataset</span>


This notebook completes the entire raster processing pipeline. The output files for each dataset (CER, COT, CWP) are saved in the `resampled_rasters_combined` directory. The processed files are resampled to a consistent extent, resolution, and CRS, and then combined by date.


### Libraries

In [ ]:
import os
import re
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from tqdm import tqdm

## <span style="color: teal;">MODIS Resampled Combined Dataset</span>

### <span style="color: teal;">Setting Directories and Parameters</span>


In [ ]:
# List of base directories containing the raster files
base_dirs = {
    "CER": "Data_original/CER_merged",
    "COT": "Data_original/COT_merged",
    "CWP": "Data_original/CWP_merged"
}

# Output folder for resampled and combined files
output_folder = "Modis_resampled_combined"
os.makedirs(output_folder, exist_ok=True)

# Target extent and resolution
xmin, ymin, xmax, ymax = 19.3416, 34.6996, 28.3085, 41.7658
target_resolution = 0.01  

target_width = int((xmax - xmin) / target_resolution)
target_height = int((ymax - ymin) / target_resolution)

target_crs = 'EPSG:4326'

### <span style="color: teal;">Extracting Date from Filename  </span>

In [ ]:
# Function to extract the date from the filename

# The original MODIS filename follows the format:
# "cropped_modis_MOD06_L2.A2017111.0855.061.2017314174119_HEGOUT.tif"
# where:
# - "A2017111" is the date in the format "Ayyyyddd"
def extract_date(filename):
    match = re.search(r'A(\d{7})', filename)  
    if match:
        return match.group(1)  # Return the date
    return None

### <span style="color: teal;">Resampling Raster Function  </span>

In [ ]:
# Function to resample a raster file
def resample_raster(input_file, output_file):
    with rasterio.open(input_file) as src:
        # Read the no-data value from the source file
        nodata = src.nodata
        
        # Define the transform for the target extent and resolution
        transform = from_origin(xmin, ymax, target_resolution, target_resolution)
        
        # Create an empty array (or an array with zeros using the zeros function) for the reprojected data
        reprojected_array = np.empty((target_height, target_width), dtype=src.dtypes[0])
        
        # Reproject the raster
        reproject(
            source=rasterio.band(src, 1),
            destination=reprojected_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=target_crs,
            resampling=Resampling.bilinear,
            src_nodata=nodata,
            dst_nodata=nodata
        )
        
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        
        # Save the resampled file with the same data type as the original
        with rasterio.open(
            output_file,
            'w',
            driver='GTiff',
            count=1,
            dtype=src.dtypes[0],  # Use the original data type!!!
            crs=target_crs,
            transform=transform,
            width=target_width,
            height=target_height,
            nodata=nodata  # Preserve the no-data value
        ) as dst:
            dst.write(reprojected_array, 1)

### <span style="color: teal;">Combining Multiple TIFFs by Date  </span>

In some cases, there were two MODIS images for the same date because one image did not cover the entire area of Greece.
In these cases, we averaged the overlapping pixels between the two images to create a single dataset. This helps to ensure that the entire region is covered and reduces the impact of gaps or missing data.

In [ ]:
# Function to combine multiple TIFFs for the same date
def combine_tiffs_for_date(tif_files, output_file):
    datasets = []
    for tif_file in tif_files:
        try:
            dataset = rasterio.open(tif_file)
            datasets.append(dataset)
        except Exception as e:
            print(f"Error opening {tif_file}: {e}")
            continue

    if not datasets:
        print(f"No valid datasets found for {output_file}")
        return

    # Read all datasets into memory
    data = []
    for dataset in datasets:
        band = dataset.read(1, masked=True)  # Read the first band
        data.append(band)

    # Stack the data and compute the mean
    stacked_data = np.ma.stack(data)
    mean_data = np.ma.mean(stacked_data, axis=0)

    # Get metadata from the first dataset
    meta = datasets[0].meta
    meta.update(count=1, dtype='int16')  # Force output data type to Int16

    # Convert the mean data back to Int16
    mean_data_filled = np.ma.filled(mean_data, np.nan)  # Replace masked values with NaN
    mean_data_int16 = np.round(mean_data_filled).astype('int16')


    # Replace no-data values
    mean_data_int16 = np.ma.filled(mean_data_int16, meta['nodata'])

    # Save the mean dataset
    with rasterio.open(
        output_file,
        'w',
        **meta
    ) as dst:
        dst.write(mean_data_int16, 1)

    # Close all datasets
    for dataset in datasets:
        dataset.close()

### <span style="color: teal;">Finding TIFF Files in Base Directory   </span>

In [ ]:
# Function to find .tif files within a given base directory
def find_tif_files(base_dir):
    tif_files = []
    if os.path.exists(base_dir):
        for root, _, files in os.walk(base_dir):
            for file in files:
                if file.lower().endswith(".tif"):
                    tif_files.append(os.path.join(root, file))
    return tif_files

### <span style="color: teal;">Main Processing Loop for Each Dataset   </span>

In [ ]:
# Process each MODIS dataset (CER, COT, CWP)
for dataset_name, base_dir in base_dirs.items():
    print(f"Processing {dataset_name}...")
    
    # Create output subfolder for the dataset
    output_subfolder = os.path.join(output_folder, dataset_name)
    os.makedirs(output_subfolder, exist_ok=True)
    
    # Find all TIFF files in the base directory
    tif_files = find_tif_files(base_dir)
    
    # Group TIFF files by date
    grouped_files = {}
    for tif_file in tif_files:
        date = extract_date(os.path.basename(tif_file))
        if date:
            if date not in grouped_files:
                grouped_files[date] = []
            grouped_files[date].append(tif_file)
    
    # Process each date group
    for date, files in tqdm(grouped_files.items(), desc=f"Combining {dataset_name} files", unit="date"):
        if len(files) > 1:
            # Resample each file first
            resampled_files = []
            for file in files:
                resampled_file = os.path.join(output_subfolder, f"resampled_{os.path.basename(file)}")
                resample_raster(file, resampled_file)
                resampled_files.append(resampled_file)
            
            # Combine the resampled files
            output_file = os.path.join(output_subfolder, f"{dataset_name.lower()}_res_A{date}.tif")
            combine_tiffs_for_date(resampled_files, output_file)
            
            # Clean up temporary resampled files
            for resampled_file in resampled_files:
                os.remove(resampled_file)
        else:
            # If only one file, resample and save it directly
            output_file = os.path.join(output_subfolder, f"{dataset_name.lower()}_res_A{date}.tif")
            resample_raster(files[0], output_file)
    
    print(f"Finished processing {dataset_name}.")

 ## <span style="color: #FFA500 ;">ERA-Wind Resampled Combined Dataset    </span>

In [ ]:
# Output folder for resampled and combined files
output_folder = "Wind_resampled_combined"
os.makedirs(output_folder, exist_ok=True)

# Target extent and resolution
xmin, ymin, xmax, ymax = 19.3416, 34.6996, 28.3085, 41.7658
target_resolution = 0.01  # Resolution in degrees (for EPSG:4326!!!)
target_crs = 'EPSG:4326'

# Path to the Wind folder
wind_folder = "Data_original/wind"

### <span style="color: #FFA500 ;">Resampling Multi-Band Raster Function (Wind)    </span>

In [ ]:
# Function to resample a multiband raster file and save each band separately
def resample_multiband_raster_(input_file, output_folder, xmin, ymin, xmax, ymax, target_resolution, target_crs):
    with rasterio.open(input_file) as src:
        # Read the no-data value from the source file
        nodata = src.nodata
        
        # Calculate the target width and height based on the target extent and resolution
        target_width = int((xmax - xmin) / target_resolution)
        target_height = int((ymax - ymin) / target_resolution)
        
        # Define the transform for the target extent and resolution
        transform = from_origin(xmin, ymax, target_resolution, target_resolution)
        
        # Create the output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)
        
        # Loop through each band in the multiband raster
        for band_idx in range(1, src.count + 1):
            # Create an empty array for the reprojected data
            reprojected_array = np.empty((target_height, target_width), dtype=src.dtypes[0])
            
            # Reproject the band
            reproject(
                source=rasterio.band(src, band_idx),
                destination=reprojected_array,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=Resampling.bilinear,
                src_nodata=nodata,
                dst_nodata=nodata
            )
            
            # Define the output file path for the resampled band
            output_file = os.path.join(output_folder, f"band_{band_idx}.tif")
            
            # Save the resampled band
            with rasterio.open(
                output_file,
                'w',
                driver='GTiff',
                count=1,
                dtype=src.dtypes[0],  # Use the original data type
                crs=target_crs,
                transform=transform,
                width=target_width,
                height=target_height,
                nodata=nodata  # Preserve the no-data value
            ) as dst:
                dst.write(reprojected_array, 1)
            
            print(f"Saved resampled band {band_idx} to {output_file}")



# Iterate through all .tif files in the Wind folder
for filename in os.listdir(wind_folder):
    if filename.endswith(".tif"):
        input_file = os.path.join(wind_folder, filename)
        
        # Create a subfolder for each resampled file
        output_subfolder = os.path.join(output_folder, os.path.splitext(filename)[0] + "_res")
        
        # Resample the raster
        resample_multiband_raster_(input_file, output_subfolder, xmin, ymin, xmax, ymax, target_resolution, target_crs)
        
        print(f"Resampled {filename} and saved to {output_subfolder}")